In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

A continuación vemos qué hace el LLM sin herramientas y vemos que da una respuesta genérica.

In [3]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Sure — usually **yes, if enrollment is still open**.  \nIf you just discovered the course, the key things to check are:\n\n- **Enrollment deadline**\n- **Whether the course has started already**\n- **Prerequisites** or required approval\n- **If there’s a waitlist**\n\nIf you want, I can help you write a message to the course instructor or admissions office asking whether you can still join.'

Ahora definimos una top-level search function que hace queries sobre el index directamente. 

In [4]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

Por lo que entiendo hasta ahora solo estamos construyendo el marco pero no le hemos pasado info realmente 

Next we tell the model about this function. The model doesn't see our Python code, only a schema describing what the function does and what arguments it takes. LLMs are language agnostic. At the end we're just making an HTTP call, so we describe the tool in JSON rather than in Python. The same schema would work from TypeScript or Java.

In [5]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [6]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join course if I just discovered the course"}', call_id='call_ShjrLLu8Ud5keS2nXOGDZmvC', name='search', type='function_call', id='fc_0a8c9c42f26232be006a4253c587588194af949018e5aa7dfb', namespace=None, status='completed')]

Dato curioso, todo el código de arriba corre incluso sin tener un index :v 

In [7]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [8]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [9]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes, you can still join.\n\nIf you want a certificate, you’ll need to submit your project while submissions are still open.'

# THE AGENTIC LOOP

In [10]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [11]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [12]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course late enrollment can I join the course discovered the course"}
function_call: search {"query":"late join course enrollment discovered course FAQ"}


In [13]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join. The course materials are available, and you can start learning whenever you want.

One important note: if you want a certificate, you need to submit your project while the course is still accepting submissions. For certificates, you also need to finish with the live cohort rather than only self-paced.

If you want, I can also help you with the best way to get started or explain the certificate requirements in more detail.


In [14]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [15]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama local run install run locally Ollama"}
iteration #2...
function_call: search {"query":"ollama serve localhost 11434 run llama3 local server course FAQ"}
iteration #3...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - macOS: download the `.pkg` from https://ollama.com/download
   - Windows: download the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a local model**
   ```bash
   ollama run llama3
   ```
   This downloads the model, starts it locally, and opens a chat-like terminal interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response from Ollama.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

   print(response['message

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - macOS: download the `.pkg` from https://ollama.com/download\n   - Windows: download the `.msi`\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a local model**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model, starts it locally, and opens a chat-like terminal interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response from Ollama.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a **connection refused** error, restart the server with:\n```bash\nollama serve\n```\nor, in a notebook:\n```bash\n!nohup ollama serve > nohup.out 2>&1 &\n

Watch what happens. The agent searches for "Olama" and gets poor results. It then searches again with "Ollama" and finds the answer. The loop lets the model recover from a bad search on its own. That's the whole point of going agentic.

In [16]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enrollment discovered the course can I still join"}
iteration #2...
ASSISTANT:
Yes — you can still join the course even if you just discovered it.

If you want to receive a certificate, the key thing is to submit your project while submissions are still being accepted.

If you'd like, I can also help with:
- how to start the course now,
- whether you can still get a certificate,
- or how the weekly workflow works.

Is there anything else you want to explore?


"Yes — you can still join the course even if you just discovered it.\n\nIf you want to receive a certificate, the key thing is to submit your project while submissions are still being accepted.\n\nIf you'd like, I can also help with:\n- how to start the course now,\n- whether you can still get a certificate,\n- or how the weekly workflow works.\n\nIs there anything else you want to explore?"

In [17]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment registration start date FAQ"}
iteration #2...
function_call: search {"query":"certificate project submission accepting submissions live cohort self-paced join course FAQ"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still open. If you’re just learning, you can start anytime using the course materials.

If you want, I can also tell you the best way to start the course or explain the certificate requirements.


'Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open. If you’re just learning, you can start anytime using the course materials.\n\nIf you want, I can also tell you the best way to start the course or explain the certificate requirements.'

In [18]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit what is queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening definition declined accepted exchange"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a chess opening that starts with:

1. **d4 d5**
2. **c4**

White offers the **c-pawn** to challenge Black’s control of the center. It’s called a “gambit” because White may temporarily give up material to get better central control and piece activity.

A few common continuations:
- **Queen’s Gambit Accepted**: Black takes the pawn with `dxc4`
- **Queen’s Gambit Declined**: Black does not take it and instead supports the center, often with `e6`

If you want, I can also explain:
- the main ideas behind the opening,
- the difference between **accepted** and **declined**, or
- how to play it as White or Black.


'The **Queen’s Gambit** is a chess opening that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the **c-pawn** to challenge Black’s control of the center. It’s called a “gambit” because White may temporarily give up material to get better central control and piece activity.\n\nA few common continuations:\n- **Queen’s Gambit Accepted**: Black takes the pawn with `dxc4`\n- **Queen’s Gambit Declined**: Black does not take it and instead supports the center, often with `e6`\n\nIf you want, I can also explain:\n- the main ideas behind the opening,\n- the difference between **accepted** and **declined**, or\n- how to play it as White or Black.'

:v

In [19]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening"}
iteration #3...
ASSISTANT:
I couldn’t find a course FAQ entry for “queen’s gambit,” so this looks off-topic or not covered in the course materials.

If you meant something specific to the course, try rephrasing with course terms or context. Are there other areas you want to explore?


'I couldn’t find a course FAQ entry for “queen’s gambit,” so this looks off-topic or not covered in the course materials.\n\nIf you meant something specific to the course, try rephrasing with course terms or context. Are there other areas you want to explore?'

# TOYAIKIT

In [20]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [21]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [22]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [23]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [24]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [25]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [26]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [27]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


Continuing the conversation

Se pasa el *messages* de los resultados previos y los pasas a loop como "previous_messages"

In [ ]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)